In [1]:
import json 
from urllib.request import Request, urlopen 
 
TEAM_ID = "TEAM_27" 
API_KEY = "oc_pdpM6GDIce6DojNtkSOchRwtO3Z0ZXfE" 
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle" 
 
def api(payload): 
    req = Request(API_URL, data=json.dumps(payload).encode(), 
                  headers={"Content-Type":"application/json", "X-API-Key":API_KEY}) 
    with urlopen(req, timeout=30) as r: 
        return json.load(r) 
 
spec = api({"action":"spec", "team_id":TEAM_ID}) 
SPACE = spec["hyperparameters"] 
print("Model:", spec["model"]) 
for name, values in SPACE.items(): 
    print(name, ":", values)

Model: ElasticNet
tol : [0.001, 0.0001]
alpha : [0.0001, 0.00020691380811147902, 0.00042813323987193956, 0.0008858667904100822, 0.0018329807108324356, 0.00379269019073225, 0.007847599703514606, 0.01623776739188721, 0.03359818286283781, 0.06951927961775606, 0.14384498882876628, 0.29763514416313164, 0.615848211066026, 1.2742749857031321, 2.6366508987303554, 5.455594781168514, 11.288378916846883, 23.357214690901213, 48.32930238571752, 100]
l1_ratio : [0.05, 0.17857142857142855, 0.3071428571428571, 0.43571428571428567, 0.5642857142857143, 0.6928571428571428, 0.8214285714285714, 0.95]
positive : [True, False]
selection : ['cyclic', 'random']
fit_intercept : [True, False]


2 × 20 × 8 × 2 × 2 × 2 = 2,560 possible configurations.

Bayesian optimization

In [2]:
# ===================== CELL 2 : definitions only (makes NO API calls) =====================
# Pure Python. No numpy / pandas / math / random / itertools. Only json + os (to save the query log).
#
# SEARCH METHOD: Bayesian Optimization (sequential model-based optimization)
#   surrogate model : Gaussian Process regression (Matern-5/2 kernel), written from scratch
#   acquisition     : Expected Improvement (EI)
#   initial design  : deterministic max-min-distance (space-filling) points - no randomness
import json, os

# ---------------------------------------------------------------- tiny math helpers
_E, _LN2 = 2.718281828459045, 0.6931471805599453
def _exp(x):  return _E ** x
def _sqrt(x): return x ** 0.5
def _log(x):
    k = 0
    while x > 1.5: x /= 2.0; k += 1
    while x < 0.75: x *= 2.0; k -= 1
    y = (x - 1.0) / (x + 1.0); y2 = y * y; term, s = y, 0.0
    for n in range(1, 30, 2):
        s += term / n; term *= y2
    return 2.0 * s + k * _LN2
def _erf(x):                                   # Abramowitz-Stegun 7.1.26 (error < 1.5e-7)
    sgn = -1.0 if x < 0 else 1.0; x = abs(x)
    t = 1.0 / (1.0 + 0.3275911 * x)
    poly = t * (0.254829592 + t * (-0.284496736 + t * (1.421413741 + t * (-1.453152027 + t * 1.061405429))))
    return sgn * (1.0 - poly * _exp(-x * x))
_SQRT2, _SQRT2PI = 2.0 ** 0.5, (2.0 * 3.141592653589793) ** 0.5

# ---------------------------------------------------------------- encoding values -> numbers
def build_encoding(space):
    """numeric list -> rank in [0,1] (fine for log-spaced grids); 2 options -> 0/1; more -> one-hot."""
    names, table, owner = list(space), {}, []
    for p, n in enumerate(names):
        vals, k = space[n], len(space[n])
        if all(isinstance(v, (int, float)) and not isinstance(v, bool) for v in vals):
            order = sorted(range(k), key=lambda i: vals[i])
            rank = {i: r for r, i in enumerate(order)}
            table[n] = [[rank[i] / max(k - 1, 1)] for i in range(k)]
        elif k == 2:
            table[n] = [[0.0], [1.0]]
        else:
            table[n] = [[1.0 if j == i else 0.0 for j in range(k)] for i in range(k)]
        owner.extend([p] * len(table[n][0]))
    return names, table, owner

def all_configs(sizes):                        # every combination, as tuples of value-indices
    out = [[]]
    for s in sizes:
        out = [c + [i] for c in out for i in range(s)]
    return [tuple(c) for c in out]

# ---------------------------------------------------------------- Gaussian Process (from scratch)
def kern(a, b, w):                             # Matern-5/2 kernel
    d2 = 0.0
    for x, y, wi in zip(a, b, w):
        t = (x - y) * wi; d2 += t * t
    r = _sqrt(5.0 * d2)
    return (1.0 + r + r * r / 3.0) * _exp(-r)

def cholesky(K):
    n = len(K); L = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i + 1):
            s = K[i][j] - sum(L[i][k] * L[j][k] for k in range(j))
            if i == j:
                if s <= 1e-12: return None
                L[i][i] = _sqrt(s)
            else:
                L[i][j] = s / L[j][j]
    return L

def solve_lower(L, b):                         # solve L v = b
    v = []
    for i in range(len(b)):
        v.append((b[i] - sum(L[i][k] * v[k] for k in range(i))) / L[i][i])
    return v

def solve_upper(L, b):                         # solve L^T v = b
    n = len(b); v = [0.0] * n
    for i in range(n - 1, -1, -1):
        v[i] = (b[i] - sum(L[k][i] * v[k] for k in range(i + 1, n))) / L[i][i]
    return v

def gp_fit(X, y, w, noise):
    n = len(X)
    K = [[kern(X[i], X[j], w) for j in range(n)] for i in range(n)]
    for i in range(n): K[i][i] += noise + 1e-8
    L = cholesky(K)
    if L is None: return None
    alpha = solve_upper(L, solve_lower(L, y))
    lml = -0.5 * sum(a * b for a, b in zip(y, alpha)) - sum(_log(L[i][i]) for i in range(n))
    return L, alpha, lml

def fit_hyper(X, y, owner, n_params):
    """choose length-scales (one per hyperparameter = ARD) and noise by maximising marginal likelihood"""
    def score(ls, nz):
        r = gp_fit(X, y, [1.0 / ls[o] for o in owner], nz)
        return -1e18 if r is None else r[2]
    best_s, best_ls, best_nz = -1e18, None, None
    for l in (0.15, 0.3, 0.5, 0.8, 1.2, 2.0):
        for nz in (1e-6, 1e-3, 1e-2, 1e-1):
            s = score([l] * n_params, nz)
            if s > best_s: best_s, best_ls, best_nz = s, [l] * n_params, nz
    if len(X) >= 2 * n_params:                 # enough data to learn which parameters matter
        for _ in range(2):
            for p in range(n_params):
                for m in (0.5, 2.0):
                    t = best_ls[:]; t[p] = min(max(t[p] * m, 0.05), 5.0)
                    s = score(t, best_nz)
                    if s > best_s: best_s, best_ls = s, t
    return best_ls, best_nz

def expected_improvement(mu, sd, best):        # for minimisation
    z = (best - mu) / sd
    cdf = 0.5 * (1.0 + _erf(z / _SQRT2))
    pdf = _exp(-0.5 * z * z) / _SQRT2PI
    return (best - mu) * cdf + sd * pdf

# ---------------------------------------------------------------- the optimizer
def optimize(oracle, space, budget=25, n_init=None, patience=10, ei_tol=1e-2, ei_stall=2e-2,
             log_file="oracle_log.json", verbose=True):
    names, table, owner = build_encoding(space)
    n_params = len(names); sizes = [len(space[n]) for n in names]
    ALL = all_configs(sizes)
    ENC = {c: [v for n, i in zip(names, c) for v in table[n][i]] for c in ALL}

    hist = []                                  # [(config, loss)]; reloaded from disk so calls are never re-spent
    if log_file and os.path.exists(log_file):
        for rec in json.load(open(log_file)):
            hist.append((tuple(space[n].index(rec["params"][n]) for n in names), rec["loss"]))
    seen = {c for c, _ in hist}

    def query(cfg):                            # <-- the ONLY place an Oracle call happens
        params = {n: space[n][i] for n, i in zip(names, cfg)}
        loss = float(oracle(params))
        hist.append((cfg, loss)); seen.add(cfg)
        if log_file:
            json.dump([{"params": {n: space[n][i] for n, i in zip(names, c)}, "loss": l}
                       for c, l in hist], open(log_file, "w"), indent=1)
        if verbose:
            print(f"call {len(hist):>3}  loss={loss:.6g}  best={min(l for _, l in hist):.6g}  {params}")

    # ---- step 1: small deterministic space-filling start (middle point, then farthest-from-everything points)
    n_init = n_init or min(6, max(4, n_params))
    if not hist and budget > 0:
        query(tuple(s // 2 for s in sizes))
    while len(hist) < min(n_init, budget):
        def dist_to_tried(c):
            return min(sum((a - b) ** 2 for a, b in zip(ENC[c], ENC[h])) for h, _ in hist)
        query(max((c for c in ALL if c not in seen), key=dist_to_tried))

    # ---- step 2: Bayesian optimization loop
    best_so_far = min(l for _, l in hist); since_best = 0; low_ei = 0
    while len(hist) < budget:
        cfgs = [c for c, _ in hist]; y = [l for _, l in hist]
        use_log = min(y) > 0 and max(y) / min(y) > 10
        yt = [_log(v) for v in y] if use_log else y[:]
        m = sum(yt) / len(yt); s = _sqrt(sum((v - m) ** 2 for v in yt) / len(yt)) or 1.0
        z = [(v - m) / s for v in yt]
        X = [ENC[c] for c in cfgs]
        ls, nz = fit_hyper(X, z, owner, n_params)
        w = [1.0 / ls[o] for o in owner]
        L, alpha, _ = gp_fit(X, z, w, nz)

        best_z, best_c, best_ei = min(z), None, -1.0
        for c in ALL:                          # score untried configs with the SURROGATE (free, no Oracle call)
            if c in seen: continue
            ks = [kern(ENC[c], xi, w) for xi in X]
            mu = sum(k * a for k, a in zip(ks, alpha))
            v = solve_lower(L, ks)
            sd = _sqrt(max(1.0 - sum(t * t for t in v), 1e-12))
            ei = expected_improvement(mu, sd, best_z)
            if ei > best_ei: best_ei, best_c = ei, c
        if best_c is None: break

        low_ei = low_ei + 1 if best_ei < ei_tol else 0
        if low_ei >= 2:
            if verbose: print("stop: expected improvement is ~0, nothing promising left")
            break
        query(best_c)                          # spend exactly ONE Oracle call on the most promising config
        cur = min(l for _, l in hist)
        since_best = 0 if cur < best_so_far - 1e-12 else since_best + 1
        best_so_far = min(best_so_far, cur)
        if since_best >= patience and best_ei < ei_stall:
            if verbose: print(f"stop: no improvement for {patience} calls and low expected gain")
            break

    bc, bl = min(hist, key=lambda t: t[1])
    return {n: space[n][i] for n, i in zip(names, bc)}, bl, hist


In [3]:
# ===================== CELL 3 : RUN THE SEARCH  -->  THIS CELL MAKES REAL ORACLE CALLS =====================
# Needs Cell 1 (TEAM_ID, API_KEY, api, SPACE) and Cell 2 to have been run first.
# Nothing runs by itself: the Oracle is only called when YOU run this cell.

def oracle_query(params):
    # safety check: never send a name/value that is not in your assigned search space
    assert set(params) == set(SPACE), "wrong parameter names"
    for n, v in params.items():
        assert v in SPACE[n], f"{v!r} is not an allowed value for {n}"
    result = api({"action": "query", "team_id": TEAM_ID, "params": params})
    return float(result["loss"])

BUDGET = 6    # maximum number of Oracle calls (the search may stop earlier). Lower = fewer calls.

best_params, best_loss, history = optimize(oracle_query, SPACE, budget=BUDGET, log_file="oracle_log.json")

call   1  loss=199.277  best=199.277  {'tol': 0.0001, 'alpha': 0.14384498882876628, 'l1_ratio': 0.5642857142857143, 'positive': False, 'selection': 'random', 'fit_intercept': False}
call   2  loss=8.89049  best=8.89049  {'tol': 0.001, 'alpha': 0.0001, 'l1_ratio': 0.05, 'positive': True, 'selection': 'cyclic', 'fit_intercept': True}
call   3  loss=211.011  best=8.89049  {'tol': 0.001, 'alpha': 100, 'l1_ratio': 0.95, 'positive': True, 'selection': 'cyclic', 'fit_intercept': False}
call   4  loss=27.228  best=8.89049  {'tol': 0.001, 'alpha': 100, 'l1_ratio': 0.05, 'positive': False, 'selection': 'random', 'fit_intercept': True}
call   5  loss=8.89045  best=8.89045  {'tol': 0.0001, 'alpha': 0.0001, 'l1_ratio': 0.95, 'positive': True, 'selection': 'random', 'fit_intercept': True}
call   6  loss=27.228  best=8.89045  {'tol': 0.0001, 'alpha': 100, 'l1_ratio': 0.95, 'positive': False, 'selection': 'cyclic', 'fit_intercept': True}


In [4]:
# ===================== CELL 4 : REPORT (makes NO API calls) =====================
names = list(SPACE)
print("Best hyperparameters:", best_params)
print("Best loss           :", best_loss)
print("Total Oracle calls  :", len(history))

print("\n#   loss          " + "  ".join(names))
for k, (cfg, loss) in enumerate(history, 1):
    print(f"{k:<4}{loss:<14.6g}" + "  ".join(str(SPACE[n][i]) for n, i in zip(names, cfg)))

cur, curve = float("inf"), []
for _, l in history:
    cur = min(cur, l); curve.append(round(cur, 6))
print("\nBest-so-far after each call:", curve)

Best hyperparameters: {'tol': 0.0001, 'alpha': 0.0001, 'l1_ratio': 0.95, 'positive': True, 'selection': 'random', 'fit_intercept': True}
Best loss           : 8.89044691235116
Total Oracle calls  : 6

#   loss          tol  alpha  l1_ratio  positive  selection  fit_intercept
1   199.277       0.0001  0.14384498882876628  0.5642857142857143  False  random  False
2   8.89049       0.001  0.0001  0.05  True  cyclic  True
3   211.011       0.001  100  0.95  True  cyclic  False
4   27.228        0.001  100  0.05  False  random  True
5   8.89045       0.0001  0.0001  0.95  True  random  True
6   27.228        0.0001  100  0.95  False  cyclic  True

Best-so-far after each call: [199.276984, 8.890488, 8.890488, 8.890488, 8.890447, 8.890447]
